# 00 — Environment and frozen protocol

Validate configured paths before any computation. All new results are written under
`EXPERIMENT4_ROOT`, defaulting to the artifact root's `experiment4` directory.

The protocol records model capability differences. Qwen3.5 receives ECG and CXR images, MedGemma is restricted
to CXR, PULSE-7B and ECG-Instruct-Llama are restricted to ECG images, and GPT-OSS-20B is a retrieval-only text
reasoner. Model comparisons are direction-specific rather than treated as an interchangeable leaderboard.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
from e4_common import environment_report, write_json
report = environment_report()
write_json(EXPERIMENT4_ROOT / 'model_provenance.json', {
    'generators': {
        'qwen35_4b': {'model_id': 'Qwen/Qwen3.5-4B', 'input': 'image+text'},
        'medgemma15_4b': {'model_id': 'google/medgemma-1.5-4b-it', 'input': 'CXR image+text only', 'gated_access': True},
        'gpt_oss_20b': {'model_id': 'openai/gpt-oss-20b', 'input': 'retrieval text only'},
        'pulse7b_ecg': {'model_id': 'PULSE-ECG/PULSE-7B', 'input': 'rendered ECG image+text',
                        'license': 'apache-2.0'},
        'ecg_instruct_llama32_11b': {
            'model_id': 'convaiinnovations/ECG-Instruct-Llama-3.2-11B-Vision',
            'input': 'rendered ECG image+text', 'license': 'llama3'},
    },
    'pretraining_patient_overlap': (
        'unverified for all foundation models; both ECG specialists disclose MIMIC-IV-ECG-derived '
        'training data, so their MIMIC evaluation is not independent external validation'),
    'reasoning_record': 'structured evidence trace and uncertainty only; hidden chain-of-thought is not requested or stored',
})
report